# D3 & D4 Research Overview
**Date**: 2026-04-21 | **Purpose**: Full landscape of what we tried, what worked, and where we're stuck

---

## 1. Task Definition

### D3 (TTT-Discover)
- **Task**: Per-goal optimization — given a specific research goal (Goel et al. 2025 style, ~127 words), train Qwen3-30B to generate a high-quality **research plan**
- **Reward**: Multi-signal universal reward (v8.1 → v9), 8-10 signals scored 1-5, weighted aggregation
- **Eval goal**: TTT research plan (AI/ML domain)
- **Reference**: Goel et al. expert plan, Opus score 20/20

### D4 (Grant Proposal)
- **Task**: Grant proposal generation — given a research goal, generate a **full grant proposal** (longer, more structured than D3 plans)
- **Reward**: Forked from D3, 12 hybrid signals (structural sanity + D3 depth signals + proposal-specific signals)
- **Eval goals**: 8 goals (6 AI/ML + 2 non-AI: Chemo Toxicity, Ecosystem Dynamics) + 4 non-STEM goals
- **Reference**: Per-goal reference proposals, Opus score 15-20/20
- **Target**: NeurIPS 2026 paper

### D3 vs D4 key differences
| Aspect | D3 | D4 |
|---|---|---|
| Output length | ~2000 tokens (plan) | ~4000 tokens (full proposal) |
| Signal count | 10 (v9) | 12 (v7) |
| Domain | Single AI/ML goal | 12 diverse goals (STEM + non-STEM) |
| Grader | Qwen3-30B (+ 235B for SA) | Same |
| Base model | Qwen3-30B-A3B | Same |
| Key pipeline | CR-v7 per-signal REINFORCE | Same (forked from D3) |

## 2. Pipeline Architecture: CR-v7

Both D3 and D4 share the same core pipeline (D4 forked from D3):

```
Per iteration:
  Phase 1: Fresh Generation (n_fresh=4)
    - Generate plans from policy (with optional buffer context)
    - Grade all signals (deterministic, temp=0)
    - Optional: entropic REINFORCE on fresh plans (train_on_fresh=True)
    
  Phase 2: Critique-Conditioned Revision (n_revise=4)
    - Select parents from buffer via UCB
    - Grader emits per-signal critique + score
    - Revision prompt = goal + plan + ALL per-signal critiques
    - Generate BoN candidates, keep best by delta_aggregate
    
  Phase 3: Per-Signal REINFORCE (RL update)
    - For each revision with |delta_i| > threshold:
      - Advantage A_i = delta_i * delta_scale
      - HER-style: recompute logprob under single-critique context_i
    - LoRA rank 32, lr 4e-5
```

### Ablation configurations
| Config | Description | What it tests |
|---|---|---|
| **MAIN** | Full pipeline (buffer + CR + fresh RL + revise RL) | Everything |
| **B4** | No RL (skip_rl_update=True), buffer + CR only | In-context learning |
| **A_fresh** | Fresh RL only (no per-signal REINFORCE on revisions) | RL without CR |
| **B1** | Zero-shot baseline | Base model capability |
| **B4_stripped** | No RL, no critique in revision prompt | Pure selection |

## 3. D3 Results Summary (for reference)

### Signal evolution: v8.1 → v9
- **v8.1**: 8 active signals (S4 disabled), 2 hard gates. AUC 0.767
- **v9**: Added S2a_formalism (formula counting) + SA_arithmetic (numerical consistency via 235B)
  - Lowered Qwen ceiling from 0.975 → 0.950
  - Opus improved from 7-8 → 9/20

### Key runs (D3, single goal: TTT)

| Run | Qwen buf_max | Opus /20 | What it shows |
|---|---|---|---|
| Reference (Goel expert) | 0.748 | **20** | Calibration target |
| B1 zero-shot | 0.597 | 7 | Base model capability |
| B4_stripped (no critique, no RL) | 0.763 | 6 | Pure buffer selection |
| B4_v9 (critique, no RL, BoN=2) | 0.950 | 9 | Critique conditioning power |
| MAIN_v9 (full RL, BoN=2) | 0.950 | 9 | RL adds nothing at BoN=2 |
| B4_bon1 (no RL, BoN=1) | 0.840 | 8 | Without selection |
| **MAIN_fresh_bon1 (RL+fresh, BoN=1)** | **0.927** | **10** | **RL's unique value: exploration** |

### Component contribution hierarchy (D3)
```
critique_conditioning >> RL_with_fresh >> BoN >> nothing
      (+0.187 Qwen)      (+0.087 Qwen)   (compensates for RL)
      (+3 Opus)           (+2 Opus)
```

### Cross-grader inversion (the fundamental problem)
- 4/8 plans beat reference on Qwen (0.782-0.815 vs 0.748) but score 5-9/20 on Opus (ref 20/20)
- Spearman rho(Qwen rank, Opus rank) = **-0.16** (slight negative correlation!)
- Fatal errors missed by Qwen: CMA-ES on full LLM weights (O(n^2) infeasible), FLOP budgets off by 8 OOM, misattributed citations

### D3 takeaways for D4
1. Critique conditioning is the dominant mechanism (+0.187), not RL
2. RL only helps exploration (fresh plan quality), not exploitation
3. RL at BoN=2 is invisible because selection compensates
4. Qwen grader has a fundamental ceiling — high Qwen score != high quality
5. Signal hardening (formalism + arithmetic) helps somewhat but doesn't close the gap

## 4. D4 Signal Evolution

### v4 → v5 → v6 → v7

| Version | Signals | Key change | Problem found |
|---|---|---|---|
| **v4** | 8 counting signals (EPSRC-specific) | Initial proposal signals | Trivially Goodharted: "4+ methods = 5/5" |
| **v5** | 10 agency-neutral signals | Added G9 scope, G10 coverage, G11 evidence | Still count-based |
| **v6** | 10 hybrid (proposal + D3 depth) | Replaced G4/G6/G11 with D3 depth signals | G6_reasoning_depth also Goodharted (syntactic complexity without semantics) |
| **v7** | **12 hybrid** (current) | Added G12 formalism + G13 risk from D3 | Structural signals saturated (always 5/5) — down-weighted to 0.04 |

### D4-v7 signal weights
| Category | Signals | Weight each | Total |
|---|---|---|---|
| Structural (saturated, sanity-check) | G1, G2, G5, G10 | 0.04 | 0.16 |
| Evidence | G3 citations, G13 risk | 0.10 | 0.20 |
| Depth (from D3) | G4 focus, G6 reasoning, G11 rigor, G12 formalism | 0.12 | 0.48 |
| Proposal-specific | G8 deliverables, G9 scope | 0.08 | 0.16 |

### Key lesson
Counting-based rubrics are trivially gamed. Depth signals (focus, reasoning, evidence rigor, formalism) are harder to game but still exploitable by Qwen3-30B's self-grading.

## 5. D4 All Runs & Results

### Phase 1: Goal & Signal Exploration (2026-04-19)

| Run | Goal | Signals | Notes |
|---|---|---|---|
| Smoke test (30B) | FoundOpt | v4 | Pipeline validation |
| Pediatric genomics pilot | NIH R21 | v5 | **Abandoned**: Qwen has no domain knowledge → structurally correct but empty (Opus 7/20) |

**Decision**: Switch from non-AI goal (genomics law) to AI goal (FoundOpt) where Qwen has real domain knowledge.

---

### Phase 2: D4-v7 FoundOpt Ablation (2026-04-20)

| Run | Config | buf_max (Qwen) | fresh_r (avg 10-24) | Opus avg /20 | Opus best /20 |
|---|---|---|---|---|---|
| MAIN | Full pipeline | 0.970 | 0.803 | 8.4 | 10 |
| B4 | No RL, CR only | 1.000 | 0.806 | **11.4** | **16** |
| A_fresh | Fresh RL only | 0.625 | — | — | 5 |
| Reference | — | — | — | **20** | **20** |

**Shocking result**: B4 (no RL) beats MAIN (full RL) on Opus (11.4 vs 8.4)!
- RL pushes policy toward Qwen-reward-maximizing templates → template collapse → all plans ~10/20
- B4 preserves base model diversity → occasionally finds genuinely good plans (16/20)
- A_fresh depth signal collapse: G12=1, G11=2 (vs MAIN/B4 G12=4-5)

---

### Phase 3: 8-Goal Generalization (2026-04-20)

8 goals × 2 configs (MAIN vs B4), 25 iterations each:

| Goal | Domain | MAIN buf_max | B4 buf_max | Notes |
|---|---|---|---|---|
| 01_foundopt | AI (optimization) | 0.970 | 1.000 | Ablation above |
| 02_foundational_rl | AI (RL) | Completed | Completed | |
| 03_probabilistic_ai | AI | ~16 iters | ~16 iters | Partial |
| 04_trustworthy_ai | AI | ~16 iters | ~15 iters | Partial |
| 05_causal_healthcare | Non-AI (health) | Completed | Completed | |
| 06_neurosymbolic | AI | Completed | Completed | |
| 07_chemo_toxicity | Non-AI (biomed) | Completed | Completed | |
| 08_ecosystem_dynamics | Non-AI (ecology) | Completed | Completed | |

---

### Phase 4: Better Reward Experiment (2026-04-20/21)

**Hypothesis**: Better reward (1-10 scale, 235B grader, FoundOpt-specific rubric) → RL gradient becomes meaningful

8 conditions tested:

| Run | Grader | Critique Mode | Iters | Qwen buf_max | Opus /20 |
|---|---|---|---|---|---|
| 30B old + full_critique + ctx (B4) | 30B | full + context | 25 | 1.000 | **16** |
| 235B full_critique (MAIN) | 235B | full critique | 25 | 0.964 | 15.5 |
| 235B full_critique (B4) | 235B | full critique | 25 | 0.938 | 15.5 |
| 235B scores_only (MAIN) | 235B | scores only | 50 | 1.000 | 11 |
| 235B scores_only (B4) | 235B | scores only | 50 | 0.987 | 12 |
| 30B scores_only (MAIN) | 30B | scores only | 50 | 0.791 | 4 |
| 30B scores_only (B4) | 30B | scores only | 50 | 0.789 | 6 |
| A_fresh_only | 30B | none | 25 | 0.625 | 5 |

## 6. Why Things Don't Work: Root Cause Analysis

### Problem 1: Goodhart's Law (Fundamental Ceiling)

**Observation**: Qwen reward consistently increases, but Opus quality does NOT correspond.

```
Qwen score ↑  ≠  Actual quality ↑

Extreme case (A_fresh_only):
  Lowest Qwen plan (p25, 0.420) → Highest Opus (8/20)
  Highest Qwen plans (5 others)  → All identical templates, Opus 4-5/20
```

This holds across ALL 8 experimental conditions — switching grader (30B→235B), changing rubric scale (5→10), removing critique — nothing breaks the Goodhart ceiling.

**Root cause**: Qwen (both 30B and 235B) rewards:
- Surface specificity (version numbers, dataset names, precise hyperparameters) regardless of correctness
- Structural completeness (all sections present, all signals addressed)
- Syntactic complexity (long justifications) without checking semantic content

Qwen does NOT detect:
- Fabricated evidence (misattributed citations, invented statistics)
- Infeasible methods (CMA-ES on 7B params, FLOP budgets off by 8 OOM)
- Template collapse (5/6 plans identical)
- Mathematical decoration without substance

---

### Problem 2: Information Leakage in Critique Mode

The CR pipeline's per-signal critique tells the policy EXACTLY what to fix:

```
full_critique: "G4_focus scored 2/5 because the proposal stacks 7 unrelated methods..."
  → Policy: rewrites to mention only 2 methods
  → Qwen: scores 5/5 (fixed!)
  → Opus: still bad (methods still shallow, just fewer)
```

This is **instruction-following, not learning**. The model doesn't develop better proposal-writing ability — it just follows the grader's rewrite instructions.

Three critique modes tested:
- `full_critique`: Opus ~15-16/20 (best practical output, but instruction-following)
- `scores_only`: Opus ~11-12/20 (HER-aligned, but still Goodharts)
- `strip_critiques`: Model can't improve (too little info)

---

### Problem 3: RL Actively Hurts When Reward is Misspecified

In D4 (unlike D3 v9), **B4 beats MAIN on Opus**:

| | D3 (v9, BoN=1) | D4 (FoundOpt) |
|---|---|---|
| MAIN Opus | **10**/20 | 8.4/20 |
| B4 Opus | 8/20 | **11.4**/20 |
| RL helps? | **Yes** (+2 Opus) | **No** (-3 Opus!) |

**Why the reversal?**
- D3: Single well-calibrated goal, v9 signal hardening, BoN=1 → RL improves exploration
- D4: 12 signals with 4 saturated (always 5/5), longer output → more Goodhart surface → RL faithfully optimizes wrong reward → template collapse
- D4 MAIN's entropic objective concentrates policy on highest-reward mode → loses diversity → all plans look the same
- B4 preserves base model diversity → occasionally finds genuinely good plans

---

### Problem 4: Signal Aggregation, Not Detection

The grader's S2_rigor/G12_formalism correctly DETECTS math weakness (assigns 1-2/5), but:
- 1/8 weight (D3) or 0.12 weight (D4) is drowned out by inflated S7/S8 or G1/G2
- Result: aggregate reward is high even when depth signals are low
- This is why Qwen buf_max can reach 0.95-1.00 while Opus scores 8-12/20

---

### Problem 5: Domain Knowledge Ceiling (D4-specific)

Qwen3-30B has limited domain knowledge outside ML:
- Pediatric genomics goal: Opus 7/20 (structurally correct but empty)
- Non-STEM goals: Need G12a_analytical_framework instead of G12_formalism
- Even in ML: fabricates citations, misattributes methods, invents statistics

---

### Summary: The Ceiling Stack

```
Layer 1 (most fundamental): Grader can't distinguish surface mimicry from depth
  ↓
Layer 2: Signal aggregation drowns out the few signals that DO detect issues  
  ↓
Layer 3: RL faithfully optimizes the wrong reward → template collapse
  ↓
Layer 4: CR pipeline leaks critique info → instruction-following, not learning
  ↓
Layer 5: Policy model lacks domain knowledge → can't write correct content even if guided
```

## 7. What Actually Works (Positive Findings)

### 7.1 Critique-Conditioned Revision (dominant mechanism)
- B4_v9 - B4_stripped = **+0.187 Qwen, +3 Opus**
- No RL needed — just showing the model what's wrong and letting it rewrite
- Best practical output quality: B4 + full_critique = Opus 16/20 (D4 FoundOpt)

### 7.2 RL Helps Exploration at BoN=1 (D3 finding)
- MAIN_fresh_bon1 - B4_bon1 = **+0.087 Qwen, +2 Opus** (D3)
- RL's unique value: weight updates improve fresh plan quality
- But: only detectable at BoN=1; at BoN=2, selection compensates
- **Does not replicate in D4**: RL hurts because reward is more Goodharted

### 7.3 Signal Hardening Helps Incrementally
- v9 (S2a + SA) lowered Qwen ceiling by 0.025, improved Opus by +1-2
- D4-v7 (G12 + G13) provides similar incremental benefit
- But: gap to reference (Opus 20/20) remains ~11 points

### 7.4 235B Grader Improves Full-Critique Mode
- 235B full_critique: Opus 15.5/20 (vs 30B full_critique: Opus 16/20 with context)
- Comparable quality but more expensive
- 235B scores_only: Opus 11-12/20 (still Goodharts without critique guidance)

## 8. Where We Stand

### Best achieved results
| Setup | Qwen | Opus /20 | Notes |
|---|---|---|---|
| D4 B4 + full_critique + context (30B) | 1.000 | **16** | Best D4 quality |
| D4 235B full_critique | 0.964 | 15.5 | 235B comparable |
| D3 MAIN_fresh_bon1 (v9) | 0.927 | 10 | Best RL result |
| Reference | ~0.748 | **20** | Target |

### The gap
- Best D4: Opus 16/20 (via B4 + full critique, NO RL)
- Reference: Opus 20/20
- Gap: **4 points** on best single plan, but averaged over plans the gap is larger
- And crucially: the 16/20 comes from instruction-following (critique leakage), not from learned policy improvement

### What we've definitively shown
1. **RL with Qwen self-grading Goodharts** — across 8 conditions, 2 grader sizes, 3 critique modes
2. **CR architecture >> RL** for practical quality — B4 matches or beats MAIN
3. **RL's unique value is exploration** (D3 only) — doesn't transfer to D4's harder setting
4. **Signal hardening is incremental** — +1-2 Opus points, doesn't close the 11-point gap
5. **Template collapse is real** — fresh RL without CR converges to single template

### Open questions
1. **Stronger grader**: Would Opus-as-grader break the ceiling? (Cost: ~$2-5/iteration vs ~$0.02 for Qwen)
2. **Hybrid grading**: Qwen for fast iteration + periodic Opus validation?
3. **Different RL objective**: Current entropic REINFORCE may be too aggressive — alternatives?
4. **Paper framing**: Is the Goodhart analysis itself the paper contribution?
5. **Multi-goal generalization**: Do the 8-goal runs show transfer, or just per-goal memorization?

## 9. Run Timeline

```
2026-04-15  D3 v8.1 signal set validated (AUC 0.767)
     |
2026-04-16  D3 CR-v5: paragraph-level edit + UCB (buf_max ~0.78)
     |
2026-04-17  D3 CR-v6: locus-based revision (0.785 Qwen, Opus 9/20)
     |      Cross-grader inversion discovered (Spearman rho = -0.16)
     |      Signal ceiling QUANTIFIED: aggregation, not detection
     |
2026-04-18  D3 CR-v7: per-signal REINFORCE (0.975 Qwen, Opus 7-8/20)
     |      v9 signal hardening (S2a + SA): ceiling → 0.950, Opus → 9/20
     |      Key finding: critique >> RL >> nothing
     |
2026-04-19  D3 Fresh-RL: RL beats no-RL at BoN=1 (+0.087, +2 Opus)
     |      ICML AI4Science paper submitted
     |      D4 launched: signal v4, FoundOpt goal, pipeline fork
     |
2026-04-20  D4 signal v4→v5→v6→v7 rapid iteration
     |      D4-v7 FoundOpt ablation: B4 beats MAIN on Opus (11.4 vs 8.4)
     |      8-goal runs launched (MAIN + B4 × 8 goals)
     |      Better Reward experiment: 8 conditions
     |
2026-04-21  Better Reward analysis complete
     |      Goodhart confirmed across ALL conditions
     |      THIS OVERVIEW
     ↓
  Next steps?
```